# 01 Check TRF-Tools Pipeline Setup

This notebook checks the TRF-Tools pipeline inputs before estimating any TRFs. It is meant to be run cell-by-cell.

In [1]:
from pathlib import Path
import sys
import pandas as pd

def find_pipeline_dir(start=Path.cwd()):
    start = Path(start).resolve()
    candidates = [start, *start.parents, start / 'analysis' / 'trf_pipeline']
    for path in candidates:
        if (path / 'alice_trf_experiment.py').exists():
            return path
    raise FileNotFoundError(f'Could not find alice_trf_experiment.py from {start}')


PIPELINE_DIR = find_pipeline_dir()
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from alice_trf_experiment import BIDS_ROOT, BIDS_SEGMENT_DURATION, SEGMENT_DURATION, WAV_SEGMENT_DURATION, alice

print(f'Pipeline directory: {PIPELINE_DIR}')
print(f'BIDS root: {BIDS_ROOT}')

INFO    :  *** AliceComprehensionTRF initialized with root /Users/yanyuwoo/Data/bids on 2026-07-08 16:44:58 ***
INFO    :  Using eelbrain 0.42.0a4, mne 1.11.0.
Pipeline directory: /Users/yanyuwoo/Desktop/mcmaster-project/alice-comprehension-neural-prediction/analysis/trf_pipeline
BIDS root: /Users/yanyuwoo/Data/bids


## Subjects and Segment Durations

`SEGMENT_DURATION` is the duration source used by the TRF pipeline. It now comes from BIDS `events.tsv`. WAV duration is shown only as a QC comparison.

In [2]:
subjects = alice.get_field_values('subject')
print(f'N subjects: {len(subjects)}')
print(subjects[:10], '...', subjects[-5:])

duration_rows = []
for segment in sorted(SEGMENT_DURATION, key=int):
    bids_duration = BIDS_SEGMENT_DURATION[segment]
    wav_duration = WAV_SEGMENT_DURATION.get(segment)
    duration_rows.append({
        'segment': segment,
        'duration_source_used_by_pipeline': 'BIDS events.tsv',
        'duration_sec': bids_duration,
        'wav_duration_sec': wav_duration,
        'bids_minus_wav_sec': bids_duration - wav_duration if wav_duration is not None else None,
    })

duration_table = pd.DataFrame(duration_rows).sort_values('segment', key=lambda s: s.astype(int))
duration_table

N subjects: 49
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10'] ... ['45', '46', '47', '48', '49']


,segment,duration_source_used_by_pipeline,duration_sec,wav_duration_sec,bids_minus_wav_sec
0,1,BIDS events.tsv,57.540612,57.540612,0.0
1,2,BIDS events.tsv,60.845193,60.845193,0.0
2,3,BIDS events.tsv,63.259433,63.259433,0.0
3,4,BIDS events.tsv,69.988571,69.988571,0.0
4,5,BIDS events.tsv,66.272540,66.272540,0.0
5,6,BIDS events.tsv,63.777551,63.777551,0.0
6,7,BIDS events.tsv,62.896848,62.896848,0.0
7,8,BIDS events.tsv,57.310612,57.310612,0.0
8,9,BIDS events.tsv,57.226145,57.226145,0.0
9,10,BIDS events.tsv,61.269660,61.269660,0.0


## Predictor Files

The first formal model uses `gammatone-8` files in BIDS derivatives.

In [3]:
predictor_dir = BIDS_ROOT / 'derivatives' / 'predictors'
predictor_rows = []
for segment in sorted(SEGMENT_DURATION, key=int):
    path = predictor_dir / f'{segment}~gammatone-8.pickle'
    predictor_rows.append({'segment': segment, 'path': str(path), 'exists': path.exists()})

predictor_table = pd.DataFrame(predictor_rows)
display(predictor_table)
assert predictor_table['exists'].all(), 'Missing gammatone-8 predictor files'

,segment,path,exists
0,1,/Users/yanyuwoo/Data/bids/derivatives/predicto...,True
1,2,/Users/yanyuwoo/Data/bids/derivatives/predicto...,True
2,3,/Users/yanyuwoo/Data/bids/derivatives/predicto...,True
3,4,/Users/yanyuwoo/Data/bids/derivatives/predicto...,True
4,5,/Users/yanyuwoo/Data/bids/derivatives/predicto...,True
5,6,/Users/yanyuwoo/Data/bids/derivatives/predicto...,True
6,7,/Users/yanyuwoo/Data/bids/derivatives/predicto...,True
7,8,/Users/yanyuwoo/Data/bids/derivatives/predicto...,True
8,9,/Users/yanyuwoo/Data/bids/derivatives/predicto...,True
9,10,/Users/yanyuwoo/Data/bids/derivatives/predicto...,True


## Events for One Subject

TRF-Tools/Eelbrain reads BrainVision raw markers. The pipeline maps raw marker strings to the clean `segment` variable.

In [4]:
subject = '01'
events = alice.load_events(subject)
print(f'Loaded {events.n_cases} events for subject {subject}')
events.head()

Loaded 12 events for subject 01


#,i_start,trigger,event,time,SOA,subject,segment,duration
0,1832,1,Stimulus/S 1,3.664,57.628,01,1,58.541
1,30646,5,Stimulus/S 5,61.292,60.896,01,5,67.273
2,61094,6,Stimulus/S 6,122.19,63.312,01,6,64.778
3,92750,7,Stimulus/S 7,185.5,70.044,01,7,63.897
4,127772,8,Stimulus/S 8,255.54,66.328,01,8,58.311
5,160936,9,Stimulus/S 9,321.87,63.828,01,9,58.226
6,192850,10,Stimulus/S 10,385.7,62.96,01,10,62.27
7,224330,11,Stimulus/S 11,448.66,57.362,01,11,57.17
8,253011,12,Stimulus/S 12,506.02,57.28,01,12,47.983
9,281651,2,Stimulus/S 2,563.3,61.31,01,2,61.845


In [5]:
print('Raw event labels:', list(events['event']))
print('Clean segment labels:', list(events['segment']))

Raw event labels: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  5'), np.str_('Stimulus/S  6'), np.str_('Stimulus/S  7'), np.str_('Stimulus/S  8'), np.str_('Stimulus/S  9'), np.str_('Stimulus/S 10'), np.str_('Stimulus/S 11'), np.str_('Stimulus/S 12'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  3'), np.str_('Stimulus/S  4')]
Clean segment labels: ['1', '5', '6', '7', '8', '9', '10', '11', '12', '2', '3', '4']


## Channel-Type Check

`AUD` should remain in raw data as a `misc` channel, not as an EEG target.

In [6]:
alice.set(subject='01', raw='0.5-20')
raw = alice.load_raw(preload=False)
channel_types = raw.get_channel_types()
print(f'N channels total: {len(raw.ch_names)}')
print(f'N EEG channels: {channel_types.count("eeg")}')
print(f'N MISC channels: {channel_types.count("misc")}')
print(f'AUD present: {"AUD" in raw.ch_names}')
if 'AUD' in raw.ch_names:
    print(f'AUD type: {raw.get_channel_types(picks=["AUD"])[0]}')

INFO    :  Raw 0.5-20: filtering for /Users/yanyuwoo/Data/bids/sub-01/eeg/sub-01_task-alice_eeg.vhdr...
N channels total: 62
N EEG channels: 61
N MISC channels: 1
AUD present: True
AUD type: misc
